# 10 - Train / dev / test split
**What it does:** <in your own words, one or two sentences>
**Needs:** notebook 09 has run (tennis_clean.matches_ab exists)
**Produces:** the table tennis_features.match_splits, the stored split definition, and separate Parquet files per split under processed/model_data/
**Run after:** 09_matches_player_ab

## Install the pinned libraries

In [ ]:
# The repo is public, so requirements.txt can be read straight from GitHub.
REQUIREMENTS_URL = "https://raw.githubusercontent.com/katiedevane/tennis-project/main/requirements.txt"
!pip install -q -r $REQUIREMENTS_URL

## Sign in and load settings from Colab Secrets
Switch on Notebook access for both secrets in this notebook.

In [ ]:
import os
from google.colab import auth, userdata

auth.authenticate_user()

PROJECT_ID = userdata.get("TENNIS_PROJECT_ID")
BUCKET_NAME = userdata.get("TENNIS_BUCKET_NAME")
REGION = "EU"        # the bucket and the BigQuery datasets must share this location
DATASETS = ["tennis_raw", "tennis_clean", "tennis_features"]
os.environ["CLOUDSDK_CORE_PROJECT"] = PROJECT_ID

assert PROJECT_ID and BUCKET_NAME, "A secret is empty: check its name and Notebook access"
print("Settings loaded from Colab Secrets.")

## Helpers

In [ ]:
import pandas as pd
from google.cloud import bigquery

bq = bigquery.Client(project=PROJECT_ID, location=REGION)

def run(sql):
    config = bigquery.QueryJobConfig(maximum_bytes_billed=1_000_000_000)   # safety cap: 1 GB per query
    return bq.query(sql, job_config=config).to_dataframe()

pd.set_option("display.max_rows", 200, "display.max_columns", 100, "display.width", 200)
AB = f"{PROJECT_ID}.tennis_clean.matches_ab"
SPLIT_TABLE = f"{PROJECT_ID}.tennis_features.match_splits"

## The split definition
A fixed split by **season**, never random (matches of the same players and seasons are closely related, and a random split would let the model train on the future). The season is the year in `tourney_id`, so a whole tournament always stays in one part.

- **warmup (2010 to 2015):** used only as history for ratings and form. Never fitting or evaluation rows (minutes are empty and serve statistics patchy in these seasons).
- **train (2016 to 2022):** fitting.
- **dev (2023 to 2024):** choosing models and settings.
- **test (2025):** evaluated once, at the end.
- **holdout_2026:** a partial season (to about June) with more missing serve statistics. Kept out of fitting, tuning and evaluation.

Walkovers stay in the table, flagged `modelling_ok = FALSE`, and are excluded from fitting and evaluation. After tuning on dev, the chosen model is refitted on train plus dev, then evaluated once on test.

In [ ]:
import json, hashlib
from datetime import datetime, timezone

SPLIT_VERSION = "v1"
SPLITS = {                      # inclusive season ranges
    "warmup":       (2010, 2015),
    "train":        (2016, 2022),
    "dev":          (2023, 2024),
    "test":         (2025, 2025),
    "holdout_2026": (2026, 2026),
}
order = list(SPLITS)
for a, b in zip(order, order[1:]):
    assert SPLITS[a][1] + 1 == SPLITS[b][0], f"gap or overlap between {a} and {b}"

config = {
    "split_version": SPLIT_VERSION,
    "rule": "by season (year in tourney_id), never random; a whole tournament stays in one part",
    "splits": {k: {"first_season": lo, "last_season": hi} for k, (lo, hi) in SPLITS.items()},
    "walkovers": "kept and flagged (modelling_ok = FALSE); excluded from fitting and evaluation",
    "refit_rule": "tune on dev, refit the chosen model on train + dev, evaluate once on test",
    "ab_salt": "ab-v1",
}
config_sha256 = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()
print("config hash:", config_sha256[:12])

## Build the split table
A lookup keyed by `match_id`, so the split is defined in exactly one place and every later table joins to it.

In [ ]:
case = "\n    ".join(f"WHEN season BETWEEN {lo} AND {hi} THEN '{name}'" for name, (lo, hi) in SPLITS.items())
sql = f"""
CREATE OR REPLACE TABLE `{SPLIT_TABLE}`
PARTITION BY DATE_TRUNC(tourney_date, MONTH)
CLUSTER BY tourney_id
OPTIONS (description = 'Split per match, by season. Definition: processed/splits/{SPLIT_VERSION}/split_definition.json')
AS
SELECT match_id, tourney_id, tourney_date, season,
  CASE
    {case}
  END AS split,
  NOT post_is_walkover AS modelling_ok,
  '{SPLIT_VERSION}' AS split_version
FROM `{AB}`
"""
job = bq.query(sql); job.result()
print("Built", SPLIT_TABLE, "| bytes processed:", f"{job.total_bytes_processed:,}")

## Checks
1. Every match has a split, and the counts per split are the expected ones.
2. No tournament is in more than one split.
3. The periods do not overlap in time (each starts after the previous one ends).
4. The parts look alike where they should and differ where we expect: the share won by player A is about 0.5 everywhere, and the missing minutes and serve statistics follow the known break (warm-up mostly empty, 2026 a little more empty than 2016 to 2025).

In [ ]:
counts = run(f"""
SELECT split, COUNT(*) AS matches, COUNT(DISTINCT tourney_id) AS tournaments,
       COUNTIF(modelling_ok) AS modelling_rows, COUNTIF(NOT modelling_ok) AS walkovers,
       MIN(season) AS first_season, MAX(season) AS last_season,
       MIN(tourney_date) AS first_date, MAX(tourney_date) AS last_date
FROM `{SPLIT_TABLE}` GROUP BY split ORDER BY first_date""")
print(counts)

# 1. counts
EXPECTED = {"warmup": 14596, "train": 15591, "dev": 4935, "test": 2490, "holdout_2026": 1248}
got = dict(zip(counts["split"], counts["matches"]))
print("expected:", EXPECTED)
assert got == EXPECTED, "Counts per split differ from the expected ones"
assert run(f"SELECT COUNTIF(split IS NULL) AS n FROM `{SPLIT_TABLE}`")["n"][0] == 0, "Some matches have no split"
assert run(f"SELECT COUNT(*) AS n FROM `{SPLIT_TABLE}`")["n"][0] == run(f"SELECT COUNT(*) AS n FROM `{AB}`")["n"][0]

# 2. one split per tournament
n_mixed = run(f"""SELECT COUNT(*) AS n FROM (SELECT tourney_id FROM `{SPLIT_TABLE}` GROUP BY 1 HAVING COUNT(DISTINCT split) > 1)""")["n"][0]
print("tournaments in more than one split:", n_mixed)
assert n_mixed == 0

# 3. no overlap in time
b = counts.set_index("split").loc[order]
for a, nxt in zip(order, order[1:]):
    assert pd.to_datetime(b.loc[a, "last_date"]) < pd.to_datetime(b.loc[nxt, "first_date"]), f"{a} and {nxt} overlap in time"
print("periods follow each other with no overlap")

# 4. do the parts look alike?
print(run(f"""
SELECT s.split, ROUND(AVG(a.a_won), 3) AS share_a_won,
       ROUND(100 * AVG(IF(a.post_a_svpt IS NULL, 1, 0)), 1) AS pct_no_serve_stats,
       ROUND(100 * AVG(IF(a.post_minutes IS NULL, 1, 0)), 1) AS pct_no_minutes,
       ROUND(100 * AVG(IF(a.a_rank IS NULL OR a.b_rank IS NULL, 1, 0)), 1) AS pct_no_rank
FROM `{SPLIT_TABLE}` s JOIN `{AB}` a USING (match_id)
GROUP BY s.split ORDER BY MIN(s.tourney_date)"""))

## Store the split definition
Written once, with the counts included. If the content ever changes, save it as a new version.

In [ ]:
import hashlib

def save_versioned(path, text, content_type="text/csv"):
    """Write a file once. If it already exists with identical content, fine; if different, stop."""
    blob = bucket.blob(path)
    new_hash = hashlib.sha256(text.encode()).hexdigest()
    if blob.exists():
        if hashlib.sha256(blob.download_as_bytes()).hexdigest() == new_hash:
            print(f"{path}: already stored, identical")
        else:
            raise RuntimeError(f"{path} exists with different content. Save this as a new version instead of overwriting.")
    else:
        blob.upload_from_string(text, content_type=content_type, if_generation_match=0)
        print(f"{path}: saved, sha256 {new_hash[:12]}")

from google.cloud import storage
bucket = storage.Client(project=PROJECT_ID).bucket(BUCKET_NAME)

doc = {"config": config, "config_sha256": config_sha256,
       "counts": counts.drop(columns=["first_date", "last_date"]).to_dict("records"),
       "table": "tennis_features.match_splits"}
text = json.dumps(doc, indent=1, sort_keys=True, default=str)
save_versioned(f"processed/splits/{SPLIT_VERSION}/split_definition.json", text, content_type="application/json")
print(text)

## Write the split files
One set of Parquet files per split, in a versioned folder (`processed/model_data/release=<id>/split=<name>/`). Modelling code reads these files, not the full table, so the test data is never loaded by accident: tuning code only reads `train` and `dev`, and only the final evaluation reads `test`.

- The warm-up seasons have no file: they are history for the features, not model data.
- This first release holds the match columns only (no engineered features yet). When the features exist, set `SOURCE` to the feature table and use a new `RELEASE`.
- If the release already exists, a fingerprint of the source table and the split config are compared with the stored manifest, and the cell stops if either changed.

In [ ]:
RELEASE = "v1"
SOURCE = AB                       # when the feature table exists, point this at it and use a new RELEASE
EXPORT_SPLITS = ["train", "dev", "test", "holdout_2026"]
PREFIX = f"processed/model_data/release={RELEASE}"
MANIFEST_PREFIX = f"manifests/model_data_{RELEASE}_"

fingerprint = run(f"SELECT CAST(BIT_XOR(FARM_FINGERPRINT(TO_JSON_STRING(t))) AS STRING) AS fp FROM `{SOURCE}` t")["fp"][0]
exists = any(True for _ in bucket.list_blobs(prefix=PREFIX + "/", max_results=1))
manifests = sorted(b.name for b in bucket.list_blobs(prefix=MANIFEST_PREFIX))

if exists:
    if not manifests:
        print("Release exists but there is no manifest to check it against. Treat it as unverified.")
    else:
        rec = json.loads(bucket.blob(manifests[0]).download_as_text())
        if rec["source_fingerprint"] != fingerprint or rec["config_sha256"] != config_sha256:
            raise RuntimeError(f"The source table or the split definition changed since release {RELEASE} was written. Use a new RELEASE.")
        print(f"Release {RELEASE} already exists and matches the source and the split definition. Nothing to do.")
else:
    for name in EXPORT_SPLITS:
        uri = f"gs://{BUCKET_NAME}/{PREFIX}/split={name}/part-*.parquet"
        bq.query(f"""
EXPORT DATA OPTIONS (uri = '{uri}', format = 'PARQUET', overwrite = false) AS
SELECT a.*, s.split, s.modelling_ok
FROM `{SOURCE}` a JOIN `{SPLIT_TABLE}` s USING (match_id)
WHERE s.split = '{name}'
ORDER BY a.tourney_date, a.match_id""").result()
        print("written:", name)

    per_split = {}
    for name in EXPORT_SPLITS:
        files = [{"file": b.name, "bytes": b.size, "sha256": hashlib.sha256(b.download_as_bytes()).hexdigest()}
                 for b in bucket.list_blobs(prefix=f"{PREFIX}/split={name}/")]
        row = counts[counts["split"] == name].iloc[0]
        per_split[name] = {"matches": int(row["matches"]), "modelling_rows": int(row["modelling_rows"]), "files": files}

    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    bucket.blob(f"{MANIFEST_PREFIX}{stamp}.json").upload_from_string(
        json.dumps({"release": RELEASE, "contents": "match columns only (no engineered features yet)",
                    "source_table": SOURCE.split(".", 1)[1], "source_fingerprint": fingerprint,
                    "split_version": SPLIT_VERSION, "config_sha256": config_sha256, "splits": per_split},
                   indent=1), content_type="application/json")
    print("Manifest written")

## Check: does each file hold only its own split?
Each Parquet file set is read back and compared with the split table: same row count, only that split's label, and only seasons inside that split's range. This is the guard against a row ending up in the wrong file.

In [ ]:
for name in EXPORT_SPLITS:
    ext = bigquery.ExternalConfig("PARQUET")
    ext.source_uris = [f"gs://{BUCKET_NAME}/{PREFIX}/split={name}/part-*.parquet"]
    cfg = bigquery.QueryJobConfig(table_definitions={"snap": ext}, maximum_bytes_billed=1_000_000_000)
    r = bq.query("SELECT COUNT(*) AS n, MIN(season) AS lo, MAX(season) AS hi, COUNT(DISTINCT split) AS n_splits, "
                 "MIN(split) AS split_label FROM snap", job_config=cfg).to_dataframe().iloc[0]
    lo, hi = SPLITS[name]
    ok = (r["n"] == EXPECTED[name] and r["lo"] >= lo and r["hi"] <= hi and r["n_splits"] == 1 and r["split_label"] == name)
    print("OK  " if ok else "FAIL", name, "| rows:", int(r["n"]), "| seasons:", int(r["lo"]), "-", int(r["hi"]))
    assert ok, f"The files for {name} do not match the split table"

## Exploration view without the test rows
Exploration and tuning read this view, which holds the warm-up, train and dev rows only. The test and holdout rows are not in it, so they cannot be looked at by accident. The final evaluation reads the test files directly, in its own notebook.

When the feature table exists, rebuild this view on top of it.

In [ ]:
VIEW = f"{PROJECT_ID}.tennis_features.model_dev_view"
sql = f"""
CREATE OR REPLACE VIEW `{VIEW}` AS
SELECT a.*, s.split, s.modelling_ok
FROM `{AB}` a JOIN `{SPLIT_TABLE}` s USING (match_id)
WHERE s.split IN ('warmup', 'train', 'dev')
"""
bq.query(sql).result()

check = run(f"SELECT split, COUNT(*) AS matches FROM `{VIEW}` GROUP BY split ORDER BY split")
print(check)
assert set(check["split"]) == {"warmup", "train", "dev"}, "The view contains rows it should not"